# درس ۱۲ - کاهش تاریخچه گفتگو با نوت‌بوک ایجنت

این نوت‌بوک نشان می‌دهد چگونه می‌توان زمینه را در گفتگوهای طولانی با استفاده از Microsoft Agent Framework مدیریت کرد. با افزایش گفتگوها، تعداد توکن‌ها افزایش می‌یابد — که در نهایت از پنجره زمینه مدل فراتر می‌رود. ما این موضوع را با **الگوی خلاصه‌سازی زمینه** و یک **نوت‌بوک ایجنت** برای حافظه پایدار حل می‌کنیم.

## آنچه خواهید آموخت:
۱. **چرا مدیریت زمینه اهمیت دارد**: درک محدودیت‌های توکن و پنجره‌های زمینه  
۲. **ایجنت‌های آگاه به زمینه**: ساخت ایجنت‌هایی که زمینه مکالمه خود را مدیریت می‌کنند  
۳. **الگوی خلاصه‌سازی زمینه**: استفاده از ابزارها برای فشرده‌سازی تاریخچه گفتگو  
۴. **نوت‌بوک ایجنت**: حافظه پایدار که در برابر کاهش زمینه مقاوم است

## پیش‌نیازها:
- راه‌اندازی Azure OpenAI با پیکربندی متغیرهای محیطی  
- آشنایی با مفاهیم پایه ایجنت‌ها از درس‌های قبلی


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## چرا مدیریت کانتکست مهم است

هر LLM دارای یک **پنجره کانتکست** محدود است — حداکثر تعداد توکنی که می‌تواند در یک درخواست پردازش کند. همان‌طور که یک مکالمه چند دور پیش می‌رود:

- **تعداد توکن‌ها به صورت خطی رشد می‌کند** با هر پیام کاربر و پاسخ دستیار.
- **توکن‌های پرامپت هزینه را تسلط می‌دهند** چون کل تاریخچه در هر دور مجدداً ارسال می‌شود.
- در نهایت مکالمه **از پنجره کانتکست فراتر می‌رود** و مدل یا کوتاه می‌کند یا خطا می‌دهد.

### راهکارهای مدیریت کانتکست

| استراتژی | عملکرد | معایب |
|---|---|---|
| **کوتاه‌سازی** | حذف قدیمی‌ترین پیام‌ها | از دست رفتن کانتکست اولیه |
| **خلاصه‌سازی** | خلاصه کردن پیام‌های قدیمی‌تر | برخی جزئیات از دست می‌روند، اما نکات کلیدی حفظ می‌شود |
| **دفترچه یادداشت / حافظه خارجی** | ذخیره نکات کلیدی خارج از مکالمه | نیازمند فراخوانی ابزار است، اما در مقابل هرگونه کاهش پایدار می‌ماند |

در این دفترچه یادداشت، ما **خلاصه‌سازی** را با یک **ابزار دفترچه یادداشت** ترکیب می‌کنیم تا عامل بتواند پیوستگی را حتی زمانی که تاریخچه مکالمه متراکم می‌شود، حفظ کند.


## ایجاد یک عامل حساس به زمینه


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## شبیه‌سازی یک مکالمه طولانی

بیایید یک مکالمه چند نوبتی را بررسی کنیم تا ببینیم چگونه زمینه جمع می‌شود. نماینده باید جزئیات کلیدی (ترجیحات، بودجه، تاریخ‌های سفر) را در طول نوبت‌ها حفظ کند و پیوستگی را نشان دهد.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

توجه کنید که چگونه عامل زمینه‌ی گفتگوهای قبلی را حفظ می‌کند — او درباره ژاپن، سوشی، معابد، عکاسی، بودجه ۳۰۰۰ دلاری، سفر تنهایی و بازه زمانی آوریل اطلاع دارد. در یک گفتگو کوتاه این روش خوب عمل می‌کند، اما با افزایش طول گفتگو ارسال مجدد تمام تاریخچه هزینه‌بر می‌شود.

بیایید گفتگو را با دورهای بیشتر ادامه دهیم تا انباشته شدن زمینه را ببینیم:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## الگوی خلاصه‌سازی زمینه

همچنان که گفتگو پیش می‌رود، می‌توانیم از یک **ابزار خلاصه‌سازی** برای فشرده کردن زمینه انباشته‌شده به قالبی مختصر استفاده کنیم. عامل این ابزار را فراخوانی می‌کند تا ترجیحات کلیدی را ثبت کند به طوری که حتی اگر پیام‌های قدیمی‌تر حذف شوند، اطلاعات اساسی حفظ شود.

این الگو بلوک ساختاری برای کاهش پیشرفته‌تر تاریخچه است:
1. عامل حقایق کلیدی را از گفتگو شناسایی می‌کند
2. ابزار خلاصه‌سازی را برای ثبت آن‌ها فراخوانی می‌کند
3. پیام‌های قدیمی‌تر می‌توانند به طور ایمن حذف شوند چون خلاصه آنچه اهمیت دارد را ثبت می‌کند

در ادامه، ابزاری به نام `summarize_preferences` تعریف می‌کنیم که عامل می‌تواند برای ثبت خلاصه‌ای فشرده از آنچه یاد گرفته است، فراخوانی کند.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## خلاصه

در این درس یاد گرفتید چگونه مدیریت زمینه را در مکالمات بلند مدت عامل‌ها با استفاده از چارچوب Microsoft Agent انجام دهید:

### مفاهیم کلیدی
- **پنجره‌های زمینه محدود هستند** — هر توکن در تاریخچه مکالمه هزینه دارد و به محدودیت حساب می‌شود.
- **ابزارهای خلاصه‌سازی** به عامل اجازه می‌دهند تا زمینه انباشته شده را به خلاصه‌های فشرده تبدیل کند، مصرف توکن را کاهش داده و در عین حال اطلاعات ضروری را حفظ کند.
- **دفترچه یادداشت عامل** حافظه خارجی پایدار فراهم می‌کند که پس از هر کاهش مکالمه باقی می‌ماند.

### آنچه ساختید
- یک **عامل آگاه به زمینه** که پیوستگی را در مکالمات چند دور حفظ می‌کند
- یک **ابزار خلاصه‌سازی** (`summarize_preferences`) که جزئیات کلیدی کاربر را به صورت فشرده ثبت می‌کند
- یک **مکالمه چند دور** که حفظ زمینه و مدیریت تغییرات را نشان می‌دهد

### کاربردهای دنیای واقعی
- **روبات‌های خدمات مشتری**: یادآوری ترجیحات در جلسات پشتیبانی طولانی
- **دستیاران شخصی**: دنبال کردن پروژه‌های در حال انجام بدون توضیح دوباره زمینه
- **معلمان آموزشی**: حفظ پیشرفت دانش‌آموز در بسیاری از تعاملات

### گام‌های بعدی
- پیاده‌سازی ابزار دفترچه یادداشت کامل با پایایی مبتنی بر فایل
- افزودن برش خودکار تاریخچه پس از خلاصه‌سازی
- ترکیب با پایگاه‌های داده برداری برای جستجوی حافظه معنایی
- ساخت عامل‌هایی که می‌توانند مکالمات را روزها بعد با زمینه کامل ادامه دهند


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری آن باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هیچ گونه سوءتفاهم یا تفسیر نادرستی که ناشی از استفاده از این ترجمه باشد، نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
